# Chapter 3 — Masking NDVI by Threshold and Value Range

This companion notebook loads the multispectral drone raster and **computes NDVI**, then creates threshold and range masks, compares NDVI with its mask, plots masked NDVI, and provides an interactive map–histogram explorer.

The interactive range slider controls **both** panels: pixels outside the selected NDVI interval become transparent on the map, while histogram bars outside the interval are muted.

It follows Chapter 2 exactly:

```python
ds = compute_indices(load_data(DATA_PATH))
ndvi = ds["NDVI"]
```

The formula used by the supplied workshop code is
**NDVI = (NIR-842 − Red-668) / (NIR-842 + Red-668)**.

Keep this notebook beside `workshop_utils.py`, and place the source raster at the `DATA_PATH` configured in that module.

In [ ]:
# Run once in a new environment, then restart the kernel if prompted.
# %pip install -r requirements.txt

## 1. Imports and display settings

In [ ]:
from pathlib import Path
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import sys

plt.style.use("seaborn-v0_8-whitegrid")
NDVI_CMAP = "YlGn"
MASK_CMAP = ListedColormap(["#e8edf2", "#12664f"])
DEFAULT_THRESHOLD = 0.30

In [ ]:
from pathlib import Path
import sys

import dask
import dask.array as dask_array
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets

from IPython.display import display, Markdown, clear_output

pio.renderers.default = "plotly_mimetype+notebook_connected"

# Start a local Dask Client so workers and task activity are visible in the
# notebook. Threads avoid spawning extra Python processes and keep the worker
# setup reliable in Windows-based Jupyter environments.
from dask.distributed import Client, LocalCluster

DASK_WORKERS = 4
cluster = LocalCluster(
    n_workers=DASK_WORKERS,
    threads_per_worker=2,
    processes=False,
    dashboard_address=":0",
)
client = Client(cluster, set_as_default=True)
print(f"Dask scheduler: {client.scheduler.address}")
print(f"Dask workers: {len(client.scheduler_info()['workers'])}")
print(f"Dask dashboard: {client.dashboard_link}")
client

# The notebooks and workshop_utils.py should remain in the same folder.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "workshop_utils.py").exists():
    raise FileNotFoundError(
        "Open Jupyter in the converted workshop folder so workshop_utils.py "
        "and requirements.txt are beside this notebook."
    )
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from plotly.subplots import make_subplots
from workshop_utils import (
    BAND_INFO, pstretch, array_to_png_b64, to_uint8, WORKSHOP_INDICES, INDEX_INFO, 
    interactive_raster_explorer, plot_raster_distribution,
    projected_plot_context, make_projected_image, apply_projected_axes, compute_indices, plot_indices, save_indices_to_tiff, 
)

# Set the path to the GeoTIFF stored on your computer.
# Expected workshop layout: notebook_folder/data/your_raster.tif
RASTER_FILENAME = "NTABRT0007-20240828-ms-cog-subset3.tif"
RASTER_PATH = NOTEBOOK_DIR / "data" / RASTER_FILENAME

# _HERE = Path(__file__).resolve().parent
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"

# Also allow the TIFF to sit beside the notebook.
if not RASTER_PATH.exists():
    local_file = NOTEBOOK_DIR / RASTER_FILENAME
    if local_file.exists():
        RASTER_PATH = local_file

if not RASTER_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find the raster at {RASTER_PATH}. "
        "Edit RASTER_FILENAME / RASTER_PATH above to match the location "
        "of your local GeoTIFF."
    )

print(f"Local raster: {RASTER_PATH.resolve()}")
print(f"Dask scheduler: {client.scheduler.address}")
print(f"Dask workers: {len(client.scheduler_info()['workers'])}")


## 2. Load the multispectral data and compute NDVI

This is the same loading and index calculation used in the supplied Chapter 2 notebook. `compute_indices` selects **Red-668** and **NIR-842** and adds NDVI to the returned `xarray.Dataset`.

In [ ]:
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "workshop_utils.py").exists():
    raise FileNotFoundError(
        "Open Jupyter in the workshop folder so workshop_utils.py and "
        "requirements.txt are beside this notebook."
    )
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from workshop_utils import DATA_PATH, load_data, compute_indices

# Same computation as Chapter 2.

# Open the GeoTIFF as lazy Dask chunks. Only chunks needed by a later
# calculation are read into memory. Keep one band per chunk to bound working RAM.
RASTER_CHUNKS = {"band": 1, "x": 512, "y": 512}
raster = rxr.open_rasterio(
    RASTER_PATH,
    masked=True,
    chunks=RASTER_CHUNKS,
    cache=False,
    lock=True,
).astype("float32")

expected_band_names = [band["var"] for band in BAND_INFO]
if raster.sizes.get("band") != len(expected_band_names):
    raster.close()
    raise ValueError(
        f"Expected {len(expected_band_names)} bands, found "
        f"{raster.sizes.get('band')} in {RASTER_PATH.name}."
    )

# The supplied image uses zero as background/NoData. Keep values lazy while
# masking those pixels and converting the band dimension to named variables.
raster = raster.assign_coords(band=expected_band_names)
raster = raster.where(np.isfinite(raster) & (raster > 0))
ds = raster.to_dataset(dim="band")
ds.attrs.update({
    "filename": RASTER_PATH.name,
    "source_path": str(RASTER_PATH.resolve()),
    "instrument": "MicaSense RedEdge-P Dual",
    "image_width_px": int(raster.sizes["x"]),
    "image_height_px": int(raster.sizes["y"]),
    "pixel_size_m": float(abs(raster.rio.resolution()[0])),
    "crs": str(raster.rio.crs),
    "transform": tuple(raster.rio.transform()),
    "bounds": tuple(float(v) for v in raster.rio.bounds()),
    "dask_chunks": str(RASTER_CHUNKS),
})
for band in BAND_INFO:
    ds[band["var"]].attrs.update({
        "long_name": band["name"],
        "centre_wavelength_nm": band["nm"],
        "fwhm_nm": band["fwhm"],
    })

ds

ds = compute_indices(ds).compute()

# ds = compute_indices(load_data(DATA_PATH))
ndvi_da = ds["NDVI"]
ndvi = ndvi_da.values.astype("float32")

# Keep only finite values in the physical NDVI interval.
ndvi[(ndvi < -1) | (ndvi > 1)] = np.nan
valid = np.isfinite(ndvi)
if not valid.any():
    raise ValueError("Computed NDVI contains no valid values in [-1, 1].")

# Derive pixel-edge bounds from xarray pixel-centre coordinates.
x = np.asarray(ds.coords["x"].values, dtype=float)
y = np.asarray(ds.coords["y"].values, dtype=float)
dx = abs(float(np.median(np.diff(x)))) if x.size > 1 else 1.0
dy = abs(float(np.median(np.diff(y)))) if y.size > 1 else 1.0
left, right = float(x.min() - dx / 2), float(x.max() + dx / 2)
bottom, top = float(y.min() - dy / 2), float(y.max() + dy / 2)
extent = [left, right, bottom, top]
# crs = getattr(ndvi_da.rio, "crs", None)

# print(f"Source raster: {Path(DATA_PATH).resolve()}")
# print("Formula: (NIR-842 - Red-668) / (NIR-842 + Red-668)")
# print(f"Shape: {ndvi.shape} | CRS: {crs} | valid pixels: {valid.sum():,}")
# print(f"NDVI range: {np.nanmin(ndvi):.3f} to {np.nanmax(ndvi):.3f}")
# ds[["red668", "nir842", "NDVI"]]

## 3. Create a threshold mask

The default rule keeps pixels with **NDVI ≥ 0.30**. Change `threshold` or switch `KEEP_ABOVE` to `False` to retain pixels below the threshold. Invalid/no-data pixels always remain excluded.

In [ ]:
threshold = DEFAULT_THRESHOLD
KEEP_ABOVE = True

if KEEP_ABOVE:
    mask = valid & (ndvi >= threshold)
    rule_label = f"NDVI ≥ {threshold:.2f}"
else:
    mask = valid & (ndvi < threshold)
    rule_label = f"NDVI < {threshold:.2f}"

masked_ndvi = np.where(mask, ndvi, np.nan)
retained_pct = 100 * mask.sum() / valid.sum()
print(f"Rule: {rule_label}")
print(f"Retained: {mask.sum():,} of {valid.sum():,} valid pixels ({retained_pct:.1f}%)")

## 4. Plot NDVI and the Boolean mask side by side

In [ ]:
vmin, vmax = np.nanpercentile(ndvi, [2, 98])
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)

im0 = axes[0].imshow(ndvi, cmap=NDVI_CMAP, vmin=vmin, vmax=vmax,
                     extent=extent, origin="upper")
axes[0].set_title("NDVI", fontsize=15, weight="bold")
fig.colorbar(im0, ax=axes[0], label="NDVI", shrink=0.86)

mask_for_plot = np.where(valid, mask.astype(float), np.nan)
im1 = axes[1].imshow(mask_for_plot, cmap=MASK_CMAP, vmin=0, vmax=1,
                     extent=extent, origin="upper")
axes[1].set_title(f"Threshold mask: {rule_label}", fontsize=15, weight="bold")
cbar = fig.colorbar(im1, ax=axes[1], ticks=[0.25, 0.75], shrink=0.86)
cbar.ax.set_yticklabels(["Excluded", "Retained"])

for ax in axes:
    ax.set_xlabel("Easting / longitude")
    ax.set_ylabel("Northing / latitude")
    ax.ticklabel_format(style="plain", useOffset=False)

plt.show()

## 5. Plot the masked NDVI

Excluded pixels are transparent. The original raster is not modified.

In [ ]:
cmap = plt.colormaps[NDVI_CMAP].copy()
cmap.set_bad(alpha=0)

fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
ax.set_facecolor("#eef1f4")
im = ax.imshow(masked_ndvi, cmap=cmap, vmin=vmin, vmax=vmax,
               origin="upper")
ax.set_title(f"Masked NDVI — retaining {rule_label}", fontsize=15, weight="bold")
ax.set_xlabel("Easting / longitude")
ax.set_ylabel("Northing / latitude")
ax.ticklabel_format(style="plain", useOffset=False)
fig.colorbar(im, ax=ax, label="NDVI", shrink=0.88)
plt.show()

## 6. Interactive NDVI map and histogram

Drag either handle on the range slider. The map shows only pixels inside the selected interval; histogram bars in that interval are highlighted. `continuous_update=False` keeps interaction responsive for large rasters.

In [ ]:
def block_mean_downsample(array, max_side=700):
    """Fast stride-based display reduction; calculations retain full resolution."""
    step = max(1, int(np.ceil(max(array.shape) / max_side)))
    return array[::step, ::step], step


display_ndvi, display_step = block_mean_downsample(ndvi)
finite_values = ndvi[valid]
data_min = float(np.nanmin(finite_values))
data_max = float(np.nanmax(finite_values))
slider_step = max(0.001, round((data_max - data_min) / 400, 3))

# Fixed bins make updates fast and keep the histogram comparable while sliding.
counts, edges = np.histogram(finite_values, bins=80, range=(data_min, data_max))
centres = (edges[:-1] + edges[1:]) / 2
widths = np.diff(edges)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("NDVI within selected range", "NDVI distribution"),
    horizontal_spacing=0.10,
)
fig.add_trace(
    go.Heatmap(
        z=display_ndvi,
        x=np.linspace(left, right, display_ndvi.shape[1]),
        y=np.linspace(top, bottom, display_ndvi.shape[0]),
        colorscale="YlGn", zmin=-1, zmax=1,
        colorbar=dict(title="NDVI", x=0.46, len=0.82),
        hovertemplate="x: %{x:.2f}<br>y: %{y:.2f}<br>NDVI: %{z:.3f}<extra></extra>",
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(
        x=centres, y=counts, width=widths,
        marker_color=["#198754"] * len(centres),
        hovertemplate="NDVI: %{x:.3f}<br>Pixels: %{y:,}<extra></extra>",
        showlegend=False,
    ),
    row=1, col=2,
)
fig.update_xaxes(title_text="Easting / longitude", row=1, col=1)
fig.update_yaxes(title_text="Northing / latitude", autorange="reversed", row=1, col=1)
fig.update_xaxes(title_text="NDVI", range=[data_min, data_max], row=1, col=2)
fig.update_yaxes(title_text="Pixel count", row=1, col=2)
fig.update_layout(
    template="plotly_white", height=560, width=1150,
    margin=dict(l=65, r=35, t=75, b=60),
    title=dict(text="Interactive NDVI range mask", x=0.5),
)

range_slider = widgets.FloatRangeSlider(
    value=(max(data_min, 0.20), min(data_max, 0.80)),
    min=data_min, max=data_max, step=slider_step,
    description="NDVI range:",
    continuous_update=False,
    readout_format=".3f",
    layout=widgets.Layout(width="850px"),
    style={"description_width": "initial"},
)
summary = widgets.HTML()
output = widgets.Output()


def render_range(change=None):
    low, high = range_slider.value
    selected_full = valid & (ndvi >= low) & (ndvi <= high)
    selected_display = np.where(
        np.isfinite(display_ndvi) & (display_ndvi >= low) & (display_ndvi <= high),
        display_ndvi,
        np.nan,
    )
    bar_colours = np.where(
        (centres >= low) & (centres <= high), "#198754", "#d9dee3"
    ).tolist()
    fig.data[0].z = selected_display
    fig.data[1].marker.color = bar_colours
    pct = 100 * selected_full.sum() / valid.sum()
    summary.value = (
        f"<b>Selected:</b> {selected_full.sum():,} pixels "
        f"({pct:.1f}% of valid pixels) &nbsp; | &nbsp; "
        f"<b>Range:</b> {low:.3f} to {high:.3f} &nbsp; | &nbsp; "
        f"<b>Display stride:</b> {display_step}"
    )
    with output:
        output.clear_output(wait=True)
        display(fig)


range_slider.observe(render_range, names="value")
display(widgets.VBox([range_slider, summary, output]))
render_range()

## 7. Optional: drag threshold lines on the histogram

NDVI is on the left and its histogram on the right. Drag the **blue** (lower) and **red** (upper) lines horizontally; the map updates to show only pixels between them. Dashed lines mark the NDVI minimum, median and maximum.

This cell uses `go.FigureWidget`, which needs the `anywidget` package (listed in `requirements.txt`). If it does not display in your environment, use the slider in Section 6 instead — it gives the same result.

In [ ]:
# Two-panel interactive figure:
# NDVI on the left and histogram on the right.
#
# Blue and red lines on the histogram are draggable (horizontally only).
# Dashed lines show the NDVI minimum, median and maximum.
#
# Recent Plotly versions may require:
# %pip install anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display


def downsample_for_display(array, max_side=700):
    """Downsample only for display; statistics use full-resolution NDVI."""
    step = max(1, int(np.ceil(max(array.shape) / max_side)))
    return array[::step, ::step]


display_ndvi = downsample_for_display(ndvi)

finite_values = ndvi[np.isfinite(ndvi)]

ndvi_min = float(np.min(finite_values))
ndvi_max = float(np.max(finite_values))
ndvi_median = float(np.median(finite_values))

# Small padding so lines at the min and max are not clipped
# by the edge of the histogram axis.
x_pad = 0.03 * (ndvi_max - ndvi_min)

# Start draggable thresholds at the 25th and 75th percentiles.
initial_low = float(np.percentile(finite_values, 25))
initial_high = float(np.percentile(finite_values, 75))

counts, edges = np.histogram(
    finite_values,
    bins=80,
    range=(ndvi_min, ndvi_max),
)

centres = (edges[:-1] + edges[1:]) / 2
widths = np.diff(edges)

fig = go.FigureWidget(
    make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "NDVI within selected range",
            "NDVI distribution",
        ),
        horizontal_spacing=0.11,
    )
)

# Initial NDVI range mask.
initial_condition = (
    np.isfinite(display_ndvi)
    & (display_ndvi >= initial_low)
    & (display_ndvi <= initial_high)
)

# Use None instead of NaN because None is JSON-compliant.
initial_map = np.where(
    initial_condition,
    display_ndvi,
    None,
).tolist()

# ------------------------------------------------------------------
# Left panel: NDVI
# ------------------------------------------------------------------

fig.add_trace(
    go.Heatmap(
        z=initial_map,
        x=np.linspace(left, right, display_ndvi.shape[1]),
        y=np.linspace(top, bottom, display_ndvi.shape[0]),
        colorscale="YlGn",
        zmin=ndvi_min,
        zmax=ndvi_max,
        colorbar=dict(
            title="NDVI",
            x=0.46,
            len=0.82,
            tickvals=[ndvi_min, ndvi_median, ndvi_max],
            ticktext=[
                f"Min {ndvi_min:.3f}",
                f"{ndvi_median:.3f}",
                f"Max {ndvi_max:.3f}",
            ],
        ),
        hovertemplate=(
            "x: %{x:.2f}<br>"
            "y: %{y:.2f}<br>"
            "NDVI: %{z:.3f}"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

# ------------------------------------------------------------------
# Right panel: histogram
# ------------------------------------------------------------------

fig.add_trace(
    go.Bar(
        x=centres,
        y=counts,
        width=widths,
        marker_color=np.where(
            (centres >= initial_low) & (centres <= initial_high),
            "#238b45",
            "#d9dee3",
        ).tolist(),
        hovertemplate=(
            "NDVI: %{x:.3f}<br>"
            "Pixels: %{y:,}"
            "<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=2,
)

# ------------------------------------------------------------------
# Draggable threshold lines
#
# Full-height vertical lines. Any tilt or vertical shift caused by
# dragging is corrected in Python, so they only move horizontally.
# ------------------------------------------------------------------

fig.add_shape(
    type="line",
    x0=initial_low,
    x1=initial_low,
    y0=0,
    y1=1,
    xref="x2",
    yref="y2 domain",
    line=dict(
        color="#1565c0",
        width=5,
    ),
    layer="above",
    editable=True,
    name="Lower threshold",
)

fig.add_shape(
    type="line",
    x0=initial_high,
    x1=initial_high,
    y0=0,
    y1=1,
    xref="x2",
    yref="y2 domain",
    line=dict(
        color="#d32f2f",
        width=5,
    ),
    layer="above",
    editable=True,
    name="Upper threshold",
)

# ------------------------------------------------------------------
# Fixed minimum, median and maximum reference lines
# ------------------------------------------------------------------

fig.add_shape(
    type="line",
    x0=ndvi_min,
    x1=ndvi_min,
    y0=0,
    y1=1,
    xref="x2",
    yref="y2 domain",
    line=dict(
        color="#6c757d",
        width=2,
        dash="dot",
    ),
    layer="above",
    editable=False,
)

fig.add_shape(
    type="line",
    x0=ndvi_median,
    x1=ndvi_median,
    y0=0,
    y1=1,
    xref="x2",
    yref="y2 domain",
    line=dict(
        color="#6f42c1",
        width=2.5,
        dash="dash",
    ),
    layer="above",
    editable=False,
)

fig.add_shape(
    type="line",
    x0=ndvi_max,
    x1=ndvi_max,
    y0=0,
    y1=1,
    xref="x2",
    yref="y2 domain",
    line=dict(
        color="#6c757d",
        width=2,
        dash="dot",
    ),
    layer="above",
    editable=False,
)

# Labels for the fixed reference lines (above the plot area).
fig.add_annotation(
    x=ndvi_min,
    y=1,
    xref="x2",
    yref="y2 domain",
    text=f"Min<br>{ndvi_min:.3f}",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(size=11, color="#495057"),
)

fig.add_annotation(
    x=ndvi_median,
    y=1,
    xref="x2",
    yref="y2 domain",
    text=f"Median<br>{ndvi_median:.3f}",
    showarrow=False,
    xanchor="center",
    yanchor="bottom",
    font=dict(size=11, color="#6f42c1"),
)

fig.add_annotation(
    x=ndvi_max,
    y=1,
    xref="x2",
    yref="y2 domain",
    text=f"Max<br>{ndvi_max:.3f}",
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    font=dict(size=11, color="#495057"),
)

# Labels for the draggable threshold lines (inside the top of the plot).
lower_label = fig.add_annotation(
    x=initial_low,
    y=0.98,
    xref="x2",
    yref="y2 domain",
    text=f"Lower: {initial_low:.3f}",
    showarrow=False,
    xanchor="right",
    yanchor="top",
    xshift=-5,
    bgcolor="rgba(255,255,255,0.85)",
    font=dict(size=11, color="#1565c0"),
)

upper_label = fig.add_annotation(
    x=initial_high,
    y=0.98,
    xref="x2",
    yref="y2 domain",
    text=f"Upper: {initial_high:.3f}",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    xshift=5,
    bgcolor="rgba(255,255,255,0.85)",
    font=dict(size=11, color="#d32f2f"),
)

fig.update_xaxes(
    title_text="Easting / longitude",
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Northing / latitude",
    autorange="reversed",
    row=1,
    col=1,
)

fig.update_xaxes(
    title_text="NDVI",
    range=[ndvi_min - x_pad, ndvi_max + x_pad],
    row=1,
    col=2,
)

# fig.update_yaxes(
#     title_text="Pixel count",
#     rangemode="tozero",
#     row=1,
#     col=2,
# )

fig.update_layout(
    template="plotly_white",
    width=1200,
    height=600,
    margin=dict(l=65, r=40, t=110, b=70),
    title=dict(
        text="Drag the blue and red lines to select the NDVI range",
        x=0.5,
    ),
)

fig._config = {
    "responsive": True,
    "displaylogo": False,
    "edits": {
        "shapePosition": True,
    },
}


# ------------------------------------------------------------------
# Keep the draggable lines vertical and full height
# ------------------------------------------------------------------

_syncing = False
_last_x = {0: initial_low, 1: initial_high}


def lock_line_vertical(index):
    """Snap a dragged threshold line back to a full-height vertical line."""
    global _syncing

    shape = fig.layout.shapes[index]
    x0 = float(shape.x0)
    x1 = float(shape.x1)
    previous = _last_x[index]

    # If one endpoint was dragged (tilting the line), follow that endpoint.
    # If the whole line was moved, both endpoints moved equally.
    new_x = x0 if abs(x0 - previous) >= abs(x1 - previous) else x1
    new_x = float(np.clip(new_x, ndvi_min, ndvi_max))
    _last_x[index] = new_x

    _syncing = True
    try:
        with fig.batch_update():
            shape.x0 = new_x
            shape.x1 = new_x
            shape.y0 = 0
            shape.y1 = 1
    finally:
        _syncing = False


def update_ndvi_from_lines(*args):
    """Update the map and histogram from the two draggable lines."""

    # The first two shapes are the draggable thresholds.
    lower_position = float(fig.layout.shapes[0].x0)
    upper_position = float(fig.layout.shapes[1].x0)

    low, high = sorted((lower_position, upper_position))

    display_condition = (
        np.isfinite(display_ndvi)
        & (display_ndvi >= low)
        & (display_ndvi <= high)
    )

    # Convert excluded pixels to None instead of NaN.
    # This prevents the jupyter-client JSON serialization warning.
    selected_map = np.where(
        display_condition,
        display_ndvi,
        None,
    ).tolist()

    full_condition = (
        np.isfinite(ndvi)
        & (ndvi >= low)
        & (ndvi <= high)
    )

    selected_pixels = int(full_condition.sum())
    valid_pixels = int(np.isfinite(ndvi).sum())
    selected_percentage = 100 * selected_pixels / valid_pixels

    bar_colours = np.where(
        (centres >= low) & (centres <= high),
        "#238b45",
        "#d9dee3",
    ).tolist()

    with fig.batch_update():
        fig.data[0].z = selected_map
        fig.data[1].marker.color = bar_colours

        # Subplot title above the NDVI panel.
        fig.layout.annotations[0].text = (
            f"NDVI: {low:.3f}–{high:.3f}"
            f"<br>{selected_pixels:,} pixels "
            f"({selected_percentage:.1f}%)"
        )

        # Annotation indices:
        # 0–1 = subplot titles
        # 2–4 = min, median and max
        # 5–6 = draggable lower and upper labels
        fig.layout.annotations[5].x = low
        fig.layout.annotations[5].text = f"Lower: {low:.3f}"

        fig.layout.annotations[6].x = high
        fig.layout.annotations[6].text = f"Upper: {high:.3f}"



# def update_ndvi_from_lines(*args):
#     """Update the map and histogram from the two draggable lines."""

#     # The first two shapes are the draggable thresholds.
#     lower_position = float(fig.layout.shapes[0].x0)
#     upper_position = float(fig.layout.shapes[1].x0)

#     low, high = sorted((lower_position, upper_position))

#     display_condition = (
#         np.isfinite(display_ndvi)
#         & (display_ndvi >= low)
#         & (display_ndvi <= high)
#     )

#     # Convert excluded pixels to None instead of NaN.
#     # This prevents the jupyter-client JSON serialization warning.
#     selected_map = np.where(
#         display_condition,
#         display_ndvi,
#         None,
#     ).tolist()

#     bar_colours = np.where(
#         (centres >= low) & (centres <= high),
#         "#238b45",
#         "#d9dee3",
#     ).tolist()

#     with fig.batch_update():
#         fig.data[0].z = selected_map
#         fig.data[1].marker.color = bar_colours

#         # Subplot title above the NDVI panel.
#         fig.layout.annotations[0].text = f"NDVI: {low:.3f}–{high:.3f}"

#         # Annotation indices:
#         # 0–1 = subplot titles
#         # 2–4 = min, median and max
#         # 5–6 = draggable lower and upper labels
#         fig.layout.annotations[5].x = low
#         fig.layout.annotations[5].text = f"Lower: {low:.3f}"

#         fig.layout.annotations[6].x = high
#         fig.layout.annotations[6].text = f"Upper: {high:.3f}"



def make_shape_handler(index):
    def handler(*args):
        if _syncing:
            return
        lock_line_vertical(index)
        update_ndvi_from_lines()
    return handler


# Observe only the first two shapes—the draggable threshold lines.
for i, threshold_shape in enumerate(fig.layout.shapes[:2]):
    threshold_shape.on_change(
        make_shape_handler(i),
        "x0",
        "x1",
        "y0",
        "y1",
    )

update_ndvi_from_lines()
display(fig)

## 8. Export the current threshold result (optional)

This writes a georeferenced single-band GeoTIFF. Masked pixels use `NaN`; edit the output path if desired.

In [ ]:
import rioxarray
EXPORT_MASKED_NDVI = True
OUTPUT_PATH = Path("outputs/masked_ndvi.tif")

if EXPORT_MASKED_NDVI:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    ndvi_export = ndvi_da.copy(data=masked_ndvi.astype("float32"))
    ndvi_export.name = "NDVI"
    ndvi_export.attrs.update({
        "long_name": "Masked Normalised Difference Vegetation Index",
        "formula": "(NIR-842 - Red-668) / (NIR-842 + Red-668)",
        "mask_rule": rule_label,
        "retained_pixels": int(mask.sum()),
        "valid_pixels": int(valid.sum()),
    })
    ndvi_export.rio.write_nodata(np.nan, inplace=True)
    ndvi_export.rio.to_raster(
        OUTPUT_PATH, dtype="float32", compress="deflate", predictor=3
    )
    print(f"Saved: {OUTPUT_PATH.resolve()}")
else:
    print("Set EXPORT_MASKED_NDVI = True to write the masked raster.")

## Interpretation notes

- A threshold mask is a classification rule, not a new spectral index.
- The best NDVI threshold depends on canopy type, illumination, soil background, phenology, and the analysis objective; inspect the histogram before fixing a scientific cutoff.
- Use the Boolean mask for counts and area summaries. Use `masked_ndvi` when you need original NDVI values only for retained pixels.
- The interactive map may be display-downsampled for speed, but selection statistics and exported values use the full-resolution raster.

In [ ]:
OUTPUT_PATH